In [1]:
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_openai import OpenAI

output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template = "List five {subject}.\n{format_instructions}",
    input_variables = ["subject"],
    partial_variables = {"format_instructions": format_instructions}
)


In [2]:
_input = prompt.format(subject="ice cream flavors")

In [3]:
print(_input)

List five ice cream flavors.
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [6]:
from langchain_community.llms import Tongyi

llm = Tongyi(
    model = "qwen-plus",
    temperature=0,
)

In [8]:
out = llm.invoke(_input)
print(out)

vanilla, chocolate, strawberry, pistachio, coffee


In [9]:
output_parser.parse(out)

['vanilla', 'chocolate', 'strawberry', 'pistachio', 'coffee']

In [1]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(model="qwen-plus",
        api_key = os.getenv("DASHSCOPE_API_KEY"),
        base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1",
        temperature=0.2, 
        max_tokens=5000,
        verbose=True
)

In [5]:
from langchain_core.pydantic_v1 import BaseModel, Field

class Joke(BaseModel):
    setup: str = Field(description="笑话的设置部分")
    punchline: str = Field(description="笑话的结尾部分")

structured_llm = llm.with_structured_output(Joke)    

/opt/anaconda3/envs/ai-spike/lib/python3.10/site-packages/langchain_openai/chat_models/base.py:1647: UserWarning: Received a Pydantic BaseModel V1 schema. This is not supported by method="json_schema". Please use method="function_calling" or specify schema via JSON Schema or Pydantic V2 BaseModel. Overriding to method="function_calling".
  warnings.warn(


In [11]:
joke = structured_llm.invoke("给我讲一个关于猫的笑话")

In [12]:
print(type(joke))
print(f"setup: {joke.setup}, punchline: {joke.punchline}")

<class '__main__.Joke'>
setup: 为什么猫喜欢在电脑上睡觉, punchline: 因为它想守住秘密，不让鼠标知道。


In [13]:
structured_llm = llm.with_structured_output(Joke, method="json_mode")

In [16]:
%%time
structured_llm.invoke("给我讲一个关于猫的笑话，用`setup`和`punchline`键以JSON形式响应")

CPU times: user 10.1 ms, sys: 2.31 ms, total: 12.4 ms
Wall time: 2.51 s


Joke(setup='为什么猫不喜欢在线购物？', punchline='因为它们害怕信用卡被刮花！')

In [19]:
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache

set_llm_cache(InMemoryCache())

structured_llm = llm.with_structured_output(Joke) 

/opt/anaconda3/envs/ai-spike/lib/python3.10/site-packages/langchain_openai/chat_models/base.py:1647: UserWarning: Received a Pydantic BaseModel V1 schema. This is not supported by method="json_schema". Please use method="function_calling" or specify schema via JSON Schema or Pydantic V2 BaseModel. Overriding to method="function_calling".
  warnings.warn(


In [20]:
print(structured_llm.invoke("给我讲一个关于猫的笑话"))

setup='为什么猫不喜欢在线购物' punchline='因为他们喜欢实物猫薄荷！'


In [21]:
print(structured_llm.invoke("给我讲一个关于猫的笑话"))

setup='为什么猫不喜欢在线购物' punchline='因为他们喜欢实物猫薄荷！'


In [22]:
print(structured_llm.invoke("给我讲一个关于猫的笑话"))

setup='为什么猫不喜欢在线购物' punchline='因为他们喜欢实物猫薄荷！'
